In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import tensorflow as tf

TRAIN_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/train_slices.csv"
VAL_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/val_slices.csv"
TEST_CSV = "/content/drive/MyDrive/ASD_XAI_Project/metadata/test_slices.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("GPU:", tf.config.list_physical_devices("GPU"))

Mounted at /content/drive
Train: 4536
Validation: 1054
Test: 1036
GPU: []


In [2]:
from tensorflow.keras import layers, models

IMG_SIZE = 224

def fire_module(x, squeeze_filters, expand_filters):
    squeeze = layers.Conv2D(
        squeeze_filters,
        kernel_size=1,
        activation="relu",
        padding="same"
    )(x)

    expand1 = layers.Conv2D(
        expand_filters,
        kernel_size=1,
        activation="relu",
        padding="same"
    )(squeeze)

    expand3 = layers.Conv2D(
        expand_filters,
        kernel_size=3,
        activation="relu",
        padding="same"
    )(squeeze)

    return layers.Concatenate()([expand1, expand3])


inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = layers.Rescaling(1.0 / 255)(inputs)

x = layers.Conv2D(
    64,
    kernel_size=3,
    strides=2,
    activation="relu",
    padding="same"
)(x)

x = layers.MaxPooling2D(
    pool_size=3,
    strides=2,
    padding="same"
)(x)

x = fire_module(x, 16, 64)
x = fire_module(x, 16, 64)

x = layers.MaxPooling2D(
    pool_size=3,
    strides=2,
    padding="same"
)(x)

x = fire_module(x, 32, 128)
x = fire_module(x, 32, 128)

x = layers.MaxPooling2D(
    pool_size=3,
    strides=2,
    padding="same"
)(x)

x = fire_module(x, 48, 192)
x = fire_module(x, 48, 192)
x = fire_module(x, 64, 256)
x = fire_module(x, 64, 256)

x = layers.Dropout(0.5)(x)

x = layers.Conv2D(
    1,
    kernel_size=1,
    activation="sigmoid"
)(x)

x = layers.GlobalAveragePooling2D()(x)

squeezenet_model = models.Model(
    inputs,
    x
)

squeezenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("SqueezeNet created!")
print(squeezenet_model.input_shape)
print(squeezenet_model.output_shape)

SqueezeNet created!
(None, 224, 224, 3)
(None, 1)


In [3]:
!cp -r "/content/drive/MyDrive/ASD_XAI_Project/slices" "/content/slices"

In [4]:
train_df["SLICE_PATH"] = train_df["SLICE_PATH"].str.replace(
    "/content/drive/MyDrive/ASD_XAI_Project/slices",
    "/content/slices",
    regex=False
)

val_df["SLICE_PATH"] = val_df["SLICE_PATH"].str.replace(
    "/content/drive/MyDrive/ASD_XAI_Project/slices",
    "/content/slices",
    regex=False
)

print(train_df["SLICE_PATH"].iloc[0])

/content/slices/Control/51482/slice_000.jpg


In [5]:
BATCH_SIZE = 16

def encode_label(label):
    return 1.0 if label == "ASD" else 0.0

def make_dataset(df, training=False):

    paths = df["SLICE_PATH"].values

    labels = np.array(
        [encode_label(x) for x in df["LABEL"].values],
        dtype=np.float32
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_png(image, channels=3)
        image = tf.image.resize(image, (224, 224))
        image = tf.cast(image, tf.float32)

        return image, label

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        ds = ds.shuffle(
            len(df),
            seed=42
        )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

print("SqueezeNet datasets ready!")

SqueezeNet datasets ready!


In [6]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = squeezenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 339s 1s/step - accuracy: 0.5026 - loss: 0.6932 - val_accuracy: 0.4991 - val_loss: 0.6930 - learning_rate: 1.0000e-04
Epoch 2/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 383s 1s/step - accuracy: 0.5046 - loss: 0.6931 - val_accuracy: 0.5009 - val_loss: 0.6933 - learning_rate: 1.0000e-04
Epoch 3/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.5445 - loss: 0.6893 - val_accuracy: 0.6233 - val_loss: 0.6840 - learning_rate: 1.0000e-04
Epoch 4/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.6292 - loss: 0.6508 - val_accuracy: 0.6015 - val_loss: 0.6977 - learning_rate: 1.0000e-04
Epoch 5/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 331s 1s/step - accuracy: 0.6764 - loss: 0.5966 - val_accuracy: 0.4953 - val_loss: 0.7138 - learning_rate: 1.0000e-04
Epoch 6/10
284/284 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.7315 - loss: 0.5557 - val_accuracy: 0.5104 - val_loss: 0.7597 - learning_rate: 5.0000e-05


In [7]:
def load_squeeze_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    image = tf.cast(image, tf.float32)
    return image


squeeze_subject_results = []

for sub_id, group in val_df.groupby("SUB_ID"):

    images = np.stack([
        load_squeeze_image(path).numpy()
        for path in group["SLICE_PATH"]
    ])

    preds = squeezenet_model.predict(
        images,
        batch_size=16,
        verbose=0
    ).reshape(-1)

    subject_probability = np.mean(preds)

    predicted_label = (
        "ASD"
        if subject_probability >= 0.5
        else "Control"
    )

    true_label = group["LABEL"].iloc[0]

    squeeze_subject_results.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "PREDICTED_LABEL": predicted_label,
        "PROBABILITY": subject_probability,
        "NUM_SLICES": len(group)
    })


squeeze_subject_df = pd.DataFrame(
    squeeze_subject_results
)

squeeze_subject_accuracy = (
    squeeze_subject_df["TRUE_LABEL"]
    == squeeze_subject_df["PREDICTED_LABEL"]
).mean()


print("=== SQUEEZENET SUBJECT-LEVEL RESULTS ===")
print(squeeze_subject_df)

print("\nSubject-level accuracy:")
print(squeeze_subject_accuracy)

=== SQUEEZENET SUBJECT-LEVEL RESULTS ===
   SUB_ID TRUE_LABEL PREDICTED_LABEL  PROBABILITY  NUM_SLICES
0   51460        ASD         Control     0.494269         176
1   51463        ASD             ASD     0.508133         176
2   51466        ASD             ASD     0.504934         176
3   51475    Control         Control     0.491068         176
4   51476    Control         Control     0.485880         174
5   51487    Control             ASD     0.500367         176

Subject-level accuracy:
0.6666666666666666


In [8]:
import os

MODEL_DIR = (
    "/content/drive/MyDrive/ASD_XAI_Project/models"
)

os.makedirs(MODEL_DIR, exist_ok=True)

SQUEEZE_PATH = (
    f"{MODEL_DIR}/squeezenet_baseline.keras"
)

squeezenet_model.save(SQUEEZE_PATH)

print("SqueezeNet saved!")
print(SQUEEZE_PATH)

SqueezeNet saved!
/content/drive/MyDrive/ASD_XAI_Project/models/squeezenet_baseline.keras


In [9]:
SQUEEZE_RESULTS_DIR = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_05_squeezenet"
)

os.makedirs(
    SQUEEZE_RESULTS_DIR,
    exist_ok=True
)

squeeze_subject_df.to_csv(
    f"{SQUEEZE_RESULTS_DIR}/squeezenet_baseline_subject_results.csv",
    index=False
)

print("SqueezeNet baseline results saved!")
print("Accuracy:", squeeze_subject_accuracy)

SqueezeNet baseline results saved!
Accuracy: 0.6666666666666666


In [10]:
!pip -q install shap

In [11]:
import shap
import numpy as np
import pandas as pd
import tensorflow as tf
import os

SQUEEZE_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/models/"
    "squeezenet_baseline.keras"
)

squeezenet_model = tf.keras.models.load_model(
    SQUEEZE_PATH
)

TRAIN_CSV = (
    "/content/drive/MyDrive/ASD_XAI_Project/metadata/"
    "train_slices.csv"
)

VAL_CSV = (
    "/content/drive/MyDrive/ASD_XAI_Project/metadata/"
    "val_slices.csv"
)

train_slices_df = pd.read_csv(TRAIN_CSV)
val_slices_df = pd.read_csv(VAL_CSV)

IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image

print("SqueezeNet loaded:", squeezenet_model.input_shape)
print("Validation slices:", len(val_slices_df))

SqueezeNet loaded: (None, 224, 224, 3)
Validation slices: 1054


In [12]:
background_images = np.stack([
    load_image(path).numpy()
    for path in train_slices_df["SLICE_PATH"].iloc[:20]
])

explainer = shap.GradientExplainer(
    squeezenet_model,
    background_images
)

print("Background:", background_images.shape)
print("SHAP explainer created!")

Background: (20, 224, 224, 3)
SHAP explainer created!


In [13]:
test_images = np.stack([
    load_image(path).numpy()
    for path in val_slices_df["SLICE_PATH"].iloc[:2]
])

test_shap = explainer.shap_values(
    test_images,
    nsamples=20
)

if isinstance(test_shap, list):
    test_shap = test_shap[0]

test_shap = np.array(test_shap)

print("Input shape:", test_images.shape)
print("SHAP shape:", test_shap.shape)
print("SqueezeNet SHAP test successful!")

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(2, 224, 224, 3))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(20, 224, 224, 3))']
  warnings.warn(msg)


Input shape: (2, 224, 224, 3)
SHAP shape: (2, 224, 224, 3, 1)
SqueezeNet SHAP test successful!


In [14]:
import os
import numpy as np
import pandas as pd

SHAP_BATCH_SIZE = 4

SQUEEZE_SHAP_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_05_squeezenet/squeezenet_shap_validation_scores.csv"
)

SQUEEZE_SHAP_CHECKPOINT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_05_squeezenet/squeezenet_shap_checkpoint.csv"
)

shap_results = []

for start in range(0, len(val_slices_df), SHAP_BATCH_SIZE):

    end = min(
        start + SHAP_BATCH_SIZE,
        len(val_slices_df)
    )

    batch_df = val_slices_df.iloc[start:end]

    batch_images = np.stack([
        load_image(path).numpy()
        for path in batch_df["SLICE_PATH"]
    ])

    shap_values = explainer.shap_values(
        batch_images,
        nsamples=20
    )

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.array(shap_values)

    if shap_values.ndim == 5:
        shap_values = np.squeeze(
            shap_values,
            axis=-1
        )

    slice_scores = np.mean(
        np.abs(shap_values),
        axis=(1, 2, 3)
    )

    for i, (_, row) in enumerate(batch_df.iterrows()):
        shap_results.append({
            "SUB_ID": row["SUB_ID"],
            "SLICE_PATH": row["SLICE_PATH"],
            "LABEL": row["LABEL"],
            "SHAP_SCORE": float(slice_scores[i])
        })

    print(
        f"SqueezeNet SHAP processed: "
        f"{end}/{len(val_slices_df)}"
    )

    if end % 100 < SHAP_BATCH_SIZE or end == len(val_slices_df):

        pd.DataFrame(shap_results).to_csv(
            SQUEEZE_SHAP_CHECKPOINT,
            index=False
        )

        print("Checkpoint saved:", end)


squeeze_shap_df = pd.DataFrame(shap_results)

squeeze_shap_df.to_csv(
    SQUEEZE_SHAP_OUTPUT,
    index=False
)

print("\nSqueezeNet SHAP completed!")
print("Rows:", len(squeeze_shap_df))

print(
    squeeze_shap_df
    .sort_values("SHAP_SCORE", ascending=False)
    .head(10)
)

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(4, 224, 224, 3))']
  warnings.warn(msg)


SqueezeNet SHAP processed: 4/1054
SqueezeNet SHAP processed: 8/1054
SqueezeNet SHAP processed: 12/1054
SqueezeNet SHAP processed: 16/1054
SqueezeNet SHAP processed: 20/1054
SqueezeNet SHAP processed: 24/1054
SqueezeNet SHAP processed: 28/1054
SqueezeNet SHAP processed: 32/1054
SqueezeNet SHAP processed: 36/1054
SqueezeNet SHAP processed: 40/1054
SqueezeNet SHAP processed: 44/1054
SqueezeNet SHAP processed: 48/1054
SqueezeNet SHAP processed: 52/1054
SqueezeNet SHAP processed: 56/1054
SqueezeNet SHAP processed: 60/1054
SqueezeNet SHAP processed: 64/1054
SqueezeNet SHAP processed: 68/1054
SqueezeNet SHAP processed: 72/1054
SqueezeNet SHAP processed: 76/1054
SqueezeNet SHAP processed: 80/1054
SqueezeNet SHAP processed: 84/1054
SqueezeNet SHAP processed: 88/1054
SqueezeNet SHAP processed: 92/1054
SqueezeNet SHAP processed: 96/1054
SqueezeNet SHAP processed: 100/1054
Checkpoint saved: 100
SqueezeNet SHAP processed: 104/1054
SqueezeNet SHAP processed: 108/1054
SqueezeNet SHAP processed: 112/1

In [15]:
import numpy as np
import pandas as pd

PERM_BATCH_SIZE = 16

SQUEEZE_PERM_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_05_squeezenet/squeezenet_permutation_validation_scores.csv"
)

permutation_results = []

rng = np.random.default_rng(42)

for start in range(0, len(val_slices_df), PERM_BATCH_SIZE):

    end = min(
        start + PERM_BATCH_SIZE,
        len(val_slices_df)
    )

    batch_df = val_slices_df.iloc[start:end]

    batch_images = np.stack([
        load_image(path).numpy()
        for path in batch_df["SLICE_PATH"]
    ])

    # Original predictions
    original_preds = squeezenet_model.predict(
        batch_images,
        batch_size=PERM_BATCH_SIZE,
        verbose=0
    ).reshape(-1)

    # Permuted images
    permuted_images = batch_images.copy()

    for i in range(len(permuted_images)):

        flat = permuted_images[i].reshape(-1).copy()
        rng.shuffle(flat)

        permuted_images[i] = flat.reshape(
            permuted_images[i].shape
        )

    # Predictions after permutation
    permuted_preds = squeezenet_model.predict(
        permuted_images,
        batch_size=PERM_BATCH_SIZE,
        verbose=0
    ).reshape(-1)

    permutation_scores = np.abs(
        original_preds - permuted_preds
    )

    for i, (_, row) in enumerate(batch_df.iterrows()):

        permutation_results.append({
            "SUB_ID": row["SUB_ID"],
            "SLICE_PATH": row["SLICE_PATH"],
            "LABEL": row["LABEL"],
            "PERMUTATION_SCORE": float(
                permutation_scores[i]
            )
        })

    print(
        f"SqueezeNet permutation processed: "
        f"{end}/{len(val_slices_df)}"
    )


squeeze_perm_df = pd.DataFrame(
    permutation_results
)

squeeze_perm_df.to_csv(
    SQUEEZE_PERM_OUTPUT,
    index=False
)

print("\nSqueezeNet permutation completed!")
print("Rows:", len(squeeze_perm_df))

print("\nTop permutation-ranked slices:")

print(
    squeeze_perm_df
    .sort_values(
        "PERMUTATION_SCORE",
        ascending=False
    )
    .head(10)
)

SqueezeNet permutation processed: 16/1054
SqueezeNet permutation processed: 32/1054
SqueezeNet permutation processed: 48/1054
SqueezeNet permutation processed: 64/1054
SqueezeNet permutation processed: 80/1054
SqueezeNet permutation processed: 96/1054
SqueezeNet permutation processed: 112/1054
SqueezeNet permutation processed: 128/1054
SqueezeNet permutation processed: 144/1054
SqueezeNet permutation processed: 160/1054
SqueezeNet permutation processed: 176/1054
SqueezeNet permutation processed: 192/1054
SqueezeNet permutation processed: 208/1054
SqueezeNet permutation processed: 224/1054
SqueezeNet permutation processed: 240/1054
SqueezeNet permutation processed: 256/1054
SqueezeNet permutation processed: 272/1054
SqueezeNet permutation processed: 288/1054
SqueezeNet permutation processed: 304/1054
SqueezeNet permutation processed: 320/1054
SqueezeNet permutation processed: 336/1054
SqueezeNet permutation processed: 352/1054
SqueezeNet permutation processed: 368/1054
SqueezeNet permut

In [16]:
squeeze_combined_df = squeeze_shap_df.merge(
    squeeze_perm_df[
        ["SUB_ID", "SLICE_PATH", "PERMUTATION_SCORE"]
    ],
    on=["SUB_ID", "SLICE_PATH"],
    how="inner"
)

# SHAP normalization
shap_min = squeeze_combined_df["SHAP_SCORE"].min()
shap_max = squeeze_combined_df["SHAP_SCORE"].max()

squeeze_combined_df["SHAP_NORM"] = (
    (squeeze_combined_df["SHAP_SCORE"] - shap_min)
    / (shap_max - shap_min)
)

# Permutation normalization
perm_min = squeeze_combined_df["PERMUTATION_SCORE"].min()
perm_max = squeeze_combined_df["PERMUTATION_SCORE"].max()

squeeze_combined_df["PERMUTATION_NORM"] = (
    (squeeze_combined_df["PERMUTATION_SCORE"] - perm_min)
    / (perm_max - perm_min)
)

print("Combined rows:", len(squeeze_combined_df))

print(
    "SHAP range:",
    squeeze_combined_df["SHAP_NORM"].min(),
    squeeze_combined_df["SHAP_NORM"].max()
)

print(
    "Permutation range:",
    squeeze_combined_df["PERMUTATION_NORM"].min(),
    squeeze_combined_df["PERMUTATION_NORM"].max()
)

Combined rows: 1054
SHAP range: 0.0 1.0
Permutation range: 0.0 1.0


In [17]:
WEIGHT_VALUES = [
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2)
]

K_VALUES = [5, 10, 15, 20]

squeeze_xai_results = []

for alpha, beta in WEIGHT_VALUES:

    temp_df = squeeze_combined_df.copy()

    temp_df["XAI_SCORE"] = (
        alpha * temp_df["SHAP_NORM"]
        + beta * temp_df["PERMUTATION_NORM"]
    )

    temp_df["XAI_RANK"] = (
        temp_df
        .groupby("SUB_ID")["XAI_SCORE"]
        .rank(
            method="first",
            ascending=False
        )
    )

    for k in K_VALUES:

        selected = temp_df[
            temp_df["XAI_RANK"] <= k
        ].copy()

        subject_results = []

        for sub_id, group in selected.groupby("SUB_ID"):

            images = np.stack([
                load_image(path).numpy()
                for path in group["SLICE_PATH"]
            ])

            predictions = squeezenet_model.predict(
                images,
                batch_size=16,
                verbose=0
            ).reshape(-1)

            probability = np.mean(predictions)

            predicted_label = (
                "ASD"
                if probability >= 0.5
                else "Control"
            )

            true_label = group["LABEL"].iloc[0]

            subject_results.append({
                "TRUE": true_label,
                "PRED": predicted_label
            })

        subject_results_df = pd.DataFrame(
            subject_results
        )

        accuracy = (
            subject_results_df["TRUE"]
            == subject_results_df["PRED"]
        ).mean()

        squeeze_xai_results.append({
            "SHAP_WEIGHT": alpha,
            "PERM_WEIGHT": beta,
            "TOP_K": k,
            "TOTAL_SLICES": len(selected),
            "ACCURACY": accuracy
        })

        print(
            f"SHAP={alpha:.1f} | "
            f"PERM={beta:.1f} | "
            f"K={k:2d} | "
            f"Accuracy={accuracy:.4f}"
        )


squeeze_xai_results_df = pd.DataFrame(
    squeeze_xai_results
)

best = squeeze_xai_results_df.loc[
    squeeze_xai_results_df["ACCURACY"].idxmax()
]

print("\n=== BEST SQUEEZENET + XAI ===")
print(best)

SHAP=0.2 | PERM=0.8 | K= 5 | Accuracy=0.5000
SHAP=0.2 | PERM=0.8 | K=10 | Accuracy=0.5000
SHAP=0.2 | PERM=0.8 | K=15 | Accuracy=0.5000
SHAP=0.2 | PERM=0.8 | K=20 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K= 5 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K=10 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K=15 | Accuracy=0.5000
SHAP=0.3 | PERM=0.7 | K=20 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K= 5 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K=10 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K=15 | Accuracy=0.5000
SHAP=0.4 | PERM=0.6 | K=20 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K= 5 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K=10 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K=15 | Accuracy=0.5000
SHAP=0.5 | PERM=0.5 | K=20 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K= 5 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K=10 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K=15 | Accuracy=0.5000
SHAP=0.6 | PERM=0.4 | K=20 | Accuracy=0.5000
SHAP=0.7 | PERM=0.3 | K= 5 | Accuracy=0.5000
SHAP=0.7 | PERM=0.3 | K=10 | Accuracy=0.5000
SHAP=0.7 |

In [18]:
SQUEEZE_XAI_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_05_squeezenet/squeezenet_xai_weight_results.csv"
)

squeeze_xai_results_df.to_csv(
    SQUEEZE_XAI_OUTPUT,
    index=False
)

print("SqueezeNet SHAP + Permutation results saved!")
print("Best XAI accuracy:", best["ACCURACY"])

SqueezeNet SHAP + Permutation results saved!
Best XAI accuracy: 0.5
